# Instruct Tuning 数据集格式

本Notebook将介绍如何构建适用于Instruct Tuning的数据集，并重点介绍Hugging Face `transformers`库中的`chat_template`方法。

一个典型的指令跟随数据集如下：

```json
[
   {"role": "system", "content": "You are a helpful assistent"},
   {"role": "user", "content": "Hello, how are you?"},
   {"role": "assistant", "content": "I'm doing great. How can I help you today?"},
   {"role": "user", "content": "I'd like to show off how chat templating works!"},
]
```
我们使用的Instruct Tuning数据集通常包含多个对话记录，每条记录包括用户输入（user input）和相应的模型回复（model response）。数据集通常以JSON格式存储。
在训练过程中，我们需要将指令跟随数据集转换为字符串，训练模型。

但是不同的模型对聊天的输入格式有着非常不同的要求。所以可以使用transformers库中的chat template方法。

## Example1

BlenderBot has an extremely simple default template, which mostly just adds whitespace between rounds of dialogue:

In [16]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("facebook/blenderbot-400M-distill")

chat = [
   {"role": "system", "content": "You are a helpful assistant"},
   {"role": "user", "content": "Hello, how are you?"},
   {"role": "assistant", "content": "I'm doing great. How can I help you today?"},
   {"role": "user", "content": "I'd like to show off how chat templating works!"},
]

print(tokenizer.apply_chat_template(chat, tokenize=False))

You are a helpful assistant   Hello, how are you?  I'm doing great. How can I help you today?   I'd like to show off how chat templating works!</s>


## Example2

Let’s use the mistralai/Mistral-7B-Instruct-v0.1 model.

In [25]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("intfloat/e5-mistral-7b-instruct")

chat = [
   {"role": "system", "content": "You are a helpful assistant"},
   {"role": "user", "content": "Hello, how are you?"},
   {"role": "assistant", "content": "I'm doing great. How can I help you today?"},
   {"role": "user", "content": "I'd like to show off how chat templating works!"},
]

print(tokenizer.apply_chat_template(chat, tokenize=False))

<s>[INST] <<SYS>>
You are a helpful assistant
<</SYS>>

Hello, how are you? [/INST] I'm doing great. How can I help you today? </s><s>[INST] I'd like to show off how chat templating works! [/INST]


## Example 3

Qwen model


In [2]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-7B-Chat")

chat = [
   {"role": "system", "content": "You are a helpful assistant"},
   {"role": "user", "content": "Hello, how are you?"},
   {"role": "assistant", "content": "I'm doing great. How can I help you today?"},
   {"role": "user", "content": "I'd like to show off how chat templating works!"},
]

print(tokenizer.apply_chat_template(chat, tokenize=False))

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
Hello, how are you?<|im_end|>
<|im_start|>assistant
I'm doing great. How can I help you today?<|im_end|>
<|im_start|>user
I'd like to show off how chat templating works!<|im_end|>



## How does it work

In [26]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-7B-Chat")

chat = [
   {"role": "system", "content": "You are a helpful assistant"},
   {"role": "user", "content": "Hello, how are you?"},
   {"role": "assistant", "content": "I'm doing great. How can I help you today?"},
   {"role": "user", "content": "I'd like to show off how chat templating works!"},
]

print('Template string')
print(tokenizer.default_chat_template)

print('-----------')
print('Instruct data')
print(tokenizer.apply_chat_template(chat, tokenize=False))

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Template string
{% for message in messages %}{{'<|im_start|>' + message['role'] + '
' + message['content'] + '<|im_end|>' + '
'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant
' }}{% endif %}
-----------
Instruct data
<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
Hello, how are you?<|im_end|>
<|im_start|>assistant
I'm doing great. How can I help you today?<|im_end|>
<|im_start|>user
I'd like to show off how chat templating works!<|im_end|>



Huggingface use Jinja template to format instruct dataset

```jinja
{% for message in messages %}
    {% if message['role'] == 'user' %}
        {{ ' ' }}
    {% endif %}
    {{ message['content'] }}
    {% if not loop.last %}
        {{ '  ' }}
    {% endif %}
{% endfor %}
{{ eos_token }}
```

This equals to:
```python
for idx, message in enumerate(messages):
    if message['role'] == 'user':
        print(' ')
    print(message['content'])
    if not idx == len(messages) - 1:  # Check for the last message in the conversation
        print('  ')
print(eos_token)
```

## Warning: problem of token boundary

Some tokenizer may have problems when dealing token boundaries. Take extra cautious when using the SentencePiece tokenizer. 

For some tokenizer:

```
tokenize(a) + tokenize(b) != tokenize(a + b)
```

In [23]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("NousResearch/Llama-2-7b-hf")

a = "Assistent: "
b = "I am a Bot"

print(f"a: {a}")
print(f"b: {b}")
print(f"a+b: {a+b}")

print(f"tokenize(a): \t\t\t{tokenizer.tokenize(a)}")
print(f"tokenize(b): \t\t\t{tokenizer.tokenize(b)}")
print(f"tokenize(a+b):\t\t\t{tokenizer.tokenize(a+b)}")
print(f"tokenize(a) + tokenize(b):\t{tokenizer.tokenize(a) + tokenizer.tokenize(b)}")

a: Assistent: 
b: I am a Bot
a+b: Assistent: I am a Bot
tokenize(a): 			['▁Ass', 'istent', ':', '▁']
tokenize(b): 			['▁I', '▁am', '▁a', '▁Bot']
tokenize(a+b):			['▁Ass', 'istent', ':', '▁I', '▁am', '▁a', '▁Bot']
tokenize(a) + tokenize(b):	['▁Ass', 'istent', ':', '▁', '▁I', '▁am', '▁a', '▁Bot']


In the inference phase, if you use 

```
"Assistent: "
```

as the prompt, then you are expecting to experience performance downgrade since the token sequence 

```
['▁Ass', 'istent', ':', '▁', '▁I', '▁am', '▁a', '▁Bot']
```

is not seen in the training